# SP-10 full-data re-run, step 2: poses for the new clips, merge, similarity (CPU, Internet ON)
**Settings:** Accelerator None, Internet ON. **Inputs:** (1) the Run 2 notebook output (`sp10_run2_data_1`), (2) the step-1 notebook output. Expect roughly 3 to 4 hours.

Extracts MediaPipe poses for the new clips only, merges them into Run 2's per-language pose files (keys must not overlap), re-audits, recomputes the similarity matrix on all train clips, and writes `manifest.json` with a sha256 for every output file.

In [ ]:
import hashlib
open('pilot_pipeline.py','w').write(r'''"""
pilot_pipeline.py (SLIM COPY for the Kaggle re-run notebooks).  Every function below is a verbatim copy of the same-named
function in pilot/pilot_pipeline.py (sha256 104edfb56e78...a74982e, 552 lines); the constants are verbatim too.  Only the
functions that sp10_driver.py and sp10_fetch_rest.py call are included: canonical_pose, resample, fill_nan_zero,
make_landmarkers, extract_video, hand_shape_vectors, location_vectors, mmd_rbf, dtw_distance, augment.  The ISLR
pilot's model, training and splitting code is not needed here and is left out.  Equality with the original is checked by
build_sp10_rerun_notebooks.py (per-function source comparison) before the notebooks are written.
"""
import os, json, math, random, zipfile, tarfile, io, time
import numpy as np

UPPER_POSE_IDX = [0, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
T_FIXED = 48
N_JOINTS = 13 + 21 + 21

def canonical_pose(pose33, lh21, rh21):
    """pose33: (T,33,3) MediaPipe normalized xyz (NaN where absent); lh21/rh21: the two hand arrays (T,21,3) in
    whatever order the extractor gave them.  Returns (T,55,3) body-centred coordinates with the hands re-ordered
    into spatial slots A (columns 13:34) and B (columns 34:55), and a (T,3) detection mask [pose, hand A, hand B]."""
    T = pose33.shape[0]
    ls, rs = pose33[:, 11, :2], pose33[:, 12, :2]                  # shoulders
    centre = np.nanmean(np.stack([ls, rs], 0), 0)                   # (T,2)
    width = np.linalg.norm(ls - rs, axis=1)                         # (T,)
    width = np.where(np.isfinite(width) & (width > 1e-3), width, np.nan)
    # fill centre/width over frames where the pose was missing
    for arr in (centre.T, width[None, :]):
        for row in arr:
            idx = np.where(np.isfinite(row))[0]
            if len(idx) == 0:
                row[:] = 0.0 if arr is centre.T else 1.0
            else:
                row[:] = np.interp(np.arange(T), idx, row[idx])
    # Hand slots are ordered by position, not by the extractor's handedness label, so that every dataset
    # (Tasks, legacy Holistic, unknown hand order) is treated identically: slot A = hand with the smaller
    # mean image-x over the frames in which it is detected; slot B = the other.
    mx = [np.nanmean(h[:, :, 0]) if np.isfinite(h[:, 0, 0]).any() else np.inf for h in (lh21, rh21)]
    hand_a, hand_b = (rh21, lh21) if mx[1] < mx[0] else (lh21, rh21)
    out = np.concatenate([pose33[:, UPPER_POSE_IDX], hand_a, hand_b], axis=1).astype(np.float32)  # (T,55,3)
    out[:, :, 0] = (out[:, :, 0] - centre[:, 0:1]) / width[:, None]
    out[:, :, 1] = (out[:, :, 1] - centre[:, 1:2]) / width[:, None]
    out[:, :, 2] = out[:, :, 2] / width[:, None]
    mask = np.stack([np.isfinite(pose33[:, 0, 0]), np.isfinite(hand_a[:, 0, 0]), np.isfinite(hand_b[:, 0, 0])], 1)
    return out, mask

def resample(seq, mask, T=T_FIXED):
    """Uniformly resample a (t,55,3) clip to T frames (nearest index); keeps NaNs."""
    t = seq.shape[0]
    idx = np.clip(np.round(np.linspace(0, t - 1, T)).astype(int), 0, t - 1)
    return seq[idx], mask[idx]

def fill_nan_zero(seq):
    """Model input: NaN -> 0 and an extra channel flag per joint saying 'observed'."""
    obs = np.isfinite(seq[..., 0]).astype(np.float32)[..., None]     # (T,55,1): per-joint observation flag, same joint order
    return np.concatenate([np.nan_to_num(seq, nan=0.0), obs], -1)

def make_landmarkers(model_dir):
    import mediapipe as mp
    from mediapipe.tasks import python as mpp
    from mediapipe.tasks.python import vision
    pose = vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(
        base_options=mpp.BaseOptions(model_asset_path=os.path.join(model_dir, "pose_landmarker_full.task")),
        running_mode=vision.RunningMode.VIDEO, num_poses=1))
    hand = vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(
        base_options=mpp.BaseOptions(model_asset_path=os.path.join(model_dir, "hand_landmarker.task")),
        running_mode=vision.RunningMode.VIDEO, num_hands=2))
    return pose, hand

_TS = {"base": 0}

def extract_video(path, pose_lm, hand_lm, max_frames=160):
    """Run pose + hand landmarkers on every frame. Returns pose33 (T,33,3), lh (T,21,3), rh (T,21,3) with NaNs."""
    import cv2, mediapipe as mp
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    if not (1.0 <= fps <= 120.0):
        fps = 25.0
    P, L, R = [], [], []
    i = 0
    base = _TS["base"]
    while True:
        ok, fr = cap.read()
        if not ok or i >= max_frames:
            break
        img = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
        ts = base + int((i + 1) * 1000 / fps)
        pr = pose_lm.detect_for_video(img, ts)
        hr = hand_lm.detect_for_video(img, ts)
        p = np.full((33, 3), np.nan, np.float32)
        if pr.pose_landmarks:
            p = np.array([[l.x, l.y, l.z] for l in pr.pose_landmarks[0]], np.float32)
        lh = np.full((21, 3), np.nan, np.float32); rh = np.full((21, 3), np.nan, np.float32)
        for lm, hd in zip(hr.hand_landmarks, hr.handedness):
            arr = np.array([[l.x, l.y, l.z] for l in lm], np.float32)
            if hd[0].category_name == "Left":
                lh = arr
            else:
                rh = arr
        P.append(p); L.append(lh); R.append(rh); i += 1
    cap.release()
    _TS["base"] = base + int((i + 2) * 1000 / fps) + 1000
    if not P:
        return None
    return np.stack(P), np.stack(L), np.stack(R)

def hand_shape_vectors(seq):
    """Per frame, per detected hand (slots A and B pooled; slot identity is irrelevant here): 21 joints relative
    to the wrist, scale-normalised -> 63-d vector.  Only detected hands contribute; missing hands are skipped,
    not zero-filled, so detection rate does not enter the hand-shape distribution."""
    vs = []
    for hs in (seq[:, 13:34], seq[:, 34:55]):
        ok = np.isfinite(hs[:, 0, 0])
        if ok.sum() == 0:
            continue
        h = hs[ok] - hs[ok][:, :1]
        scale = np.linalg.norm(h[:, 9], axis=1, keepdims=True) + 1e-6     # wrist -> middle MCP
        vs.append((h / scale[:, :, None]).reshape(len(h), -1))
    return np.concatenate(vs, 0) if vs else np.zeros((0, 63))

def location_vectors(seq):
    """Per frame: wrist positions of hand A and hand B in the body frame -> 4-d vector; NaN where absent.
    The callers keep only frames in which BOTH wrists are observed (`isfinite(...).all(1)`), so D2 describes the
    joint hand-location distribution of two-handed frames; one-handed frames do not enter D2.  Known limitation,
    recorded 2026-09-22; D1 is the primary measure."""
    return np.concatenate([seq[:, 13 + 0, :2], seq[:, 34 + 0, :2]], 1)

def mmd_rbf(X, Y, n_max=3000, seed=0):
    """Estimator (documented after the 2026-09-22 audit): the BIASED V-statistic of the SQUARED maximum mean
    discrepancy, MMD^2_b = mean K(x,x') + mean K(y,y') - 2 mean K(x,y), with diagonal terms included, using a
    Gaussian kernel K(u,v) = exp(-||u-v||^2 / h) whose bandwidth h is the median of the strictly positive squared
    pairwise distances over the pooled sample (median heuristic).  Each cloud is subsampled to at most n_max points
    with a fixed seed, so the value is deterministic.  The bias is O(1/n + 1/m), identical for every language pair
    at equal n, m, so it does not affect rankings.  Returns 0.0 when every pairwise distance is zero (identical
    degenerate clouds) instead of NaN.  Memory: two (n+m)^2 float64 matrices (d2 and K) plus the Gram product, ~0.9 GB peak at n_max=3000."""
    rng = np.random.default_rng(seed)
    X = X[rng.choice(len(X), min(n_max, len(X)), replace=False)]
    Y = Y[rng.choice(len(Y), min(n_max, len(Y)), replace=False)]
    Z = np.concatenate([X, Y]).astype(np.float64)
    sq = (Z ** 2).sum(1)
    d2 = np.maximum(sq[:, None] + sq[None, :] - 2 * Z @ Z.T, 0.0)          # (n+m)^2 matrix, no (n+m)^2 x D tensor
    pos = d2[d2 > 0]
    if pos.size == 0:                                                      # all points coincide: distributions identical
        return 0.0
    bw = float(np.median(pos)); K = np.exp(-d2 / bw)
    n, m = len(X), len(Y)
    return float(K[:n, :n].mean() + K[n:, n:].mean() - 2 * K[:n, n:].mean())

def dtw_distance(a, b, ma=None, mb=None):
    """Dynamic time warping between two sequences a (T1, J, 3) and b (T2, J, 3) with the classic recurrence
    C[i,j] = cost(i,j) + min(C[i-1,j], C[i,j-1], C[i-1,j-1]), C[0,0] = 0, borders = inf, and the accumulated cost
    normalised by T1 + T2 (an upper bound on the warping-path length; with both clips resampled to T_FIXED this is a
    constant 2*T_FIXED, so it only fixes the scale).  Design choice, kept.
    Frame cost (changed in the 2026-09-22 audit): joints missing in EITHER frame are excluded and the squared
    distance over the observed joints is rescaled by J / n_observed before the square root, so that a dataset with
    a lower hand-detection rate is not pushed away from every other language merely by its zeros.  Before the audit
    the callers passed zero-filled (T, J*3) arrays, which made missingness part of the distance.  ma, mb: (T, J)
    boolean observation masks; when omitted, all joints count.  Frames with no jointly observed joint cost 0."""
    a = np.asarray(a, np.float32); b = np.asarray(b, np.float32)
    if a.ndim == 2:                                                        # legacy flat input (T, J*3)
        a = a.reshape(len(a), -1, 3); b = b.reshape(len(b), -1, 3)
    T1, T2, J = len(a), len(b), a.shape[1]
    ma = np.ones((T1, J), bool) if ma is None else ma
    mb = np.ones((T2, J), bool) if mb is None else mb
    both = ma[:, None, :] & mb[None, :, :]                                  # (T1, T2, J)
    diff2 = ((np.nan_to_num(a)[:, None] - np.nan_to_num(b)[None]) ** 2).sum(-1) * both
    nobs = both.sum(-1)
    D = np.sqrt(diff2.sum(-1) * (J / np.maximum(nobs, 1)))
    D[nobs == 0] = 0.0
    C = np.full((T1 + 1, T2 + 1), np.inf); C[0, 0] = 0
    for i in range(1, T1 + 1):
        for j in range(1, T2 + 1):
            C[i, j] = D[i - 1, j - 1] + min(C[i - 1, j], C[i, j - 1], C[i - 1, j - 1])
    return float(C[T1, T2] / (T1 + T2))

def augment(x, rng):
    """Train-time only: small random scale/shift/rotation in the image plane, and time jitter."""
    import torch
    B = x.shape[0]
    s = torch.empty(B, 1, 1, 1).uniform_(0.85, 1.15)
    sh = torch.empty(B, 1, 1, 2).uniform_(-0.1, 0.1)
    th = torch.empty(B).uniform_(-0.2, 0.2)
    c, sn = torch.cos(th), torch.sin(th)
    xy = x[..., :2] * s
    xr = torch.stack([c[:, None, None] * xy[..., 0] - sn[:, None, None] * xy[..., 1],
                      sn[:, None, None] * xy[..., 0] + c[:, None, None] * xy[..., 1]], -1) + sh
    x = torch.cat([xr, x[..., 2:]], -1)
    # Audit 2026-09-22: the line here used to be dead code (`... if False else x`).  Its intent was correct: the
    # shift/rotation above moves zero-filled MISSING joints away from zero, which would give unobserved joints a
    # spurious coordinate signal.  Re-zero the xyz channels of unobserved joints; the observed flag (channel 3) is
    # left untouched.  Missing joints are therefore zero + flag, never imputed.
    x = torch.cat([x[..., :3] * x[..., 3:4], x[..., 3:4]], -1)
    if rng.random() < 0.5:                                           # random temporal crop + resample
        T = x.shape[1]; a = rng.randint(0, T // 8); b = T - rng.randint(0, T // 8)
        idx = torch.linspace(a, b - 1, T).round().long()
        x = x[:, idx]
    return x
''')
h = hashlib.sha256(open('pilot_pipeline.py','rb').read()).hexdigest(); print('pilot_pipeline.py', h); assert h == '0a0e4aefcbe240fb173b559cee2c59acc445530ab930ccd60fa36e789f3815e4', 'file differs from the reviewed copy'
open('sp10_driver.py','w').write(r'''"""
sp10_driver.py — SP-10 multi-parallel transfer grid: data, poses, similarity (Run 2, CPU) and the grid (Runs 3-5, GPU).
Reuses pose canonicalisation / extraction / distance functions from pilot_pipeline.py.

SP-10 (Yin et al. 2022): 10 sign languages signing the same sentences from Spreadthesign; train 830 / dev 142 / test 214
sentence ids; per sentence a video per language and the spoken-language text.  English text (lang "en", British SL) is the
common target for every source language, as in the many-to-English setting of MLSLT and in Section VIII of the paper.

Stages: A fetch metadata + clips -> B poses -> C similarity (before any training) -> D grid (one seed per run) -> E analysis.
"""
import os, sys, json, time, random, collections, subprocess, math
import numpy as np

OUT = os.environ.get("SP10_OUT", "/kaggle/working/sp10_out")
DATA = os.environ.get("SP10_DATA", "/tmp/sp10_data")
IN_POSES = os.environ.get("SP10_POSES", "")      # Runs 3-5: folder holding poses.npz + similarity.json from Run 2's output
SMOKE = int(os.environ.get("SP10_SMOKE", "0"))   # >0: only this many sentence ids per split (pipeline test)
SEED = int(os.environ.get("SP10_SEED", "0"))
EPOCHS = int(os.environ.get("SP10_EPOCHS", "30"))
LANGS = ["zh", "uk", "ru", "bg", "is", "de", "it", "sv", "lt", "en"]   # SP-10 language codes (spoken-language tags of the sign languages)
SIGN = {"zh": "CSL", "uk": "UKL", "ru": "RSL", "bg": "BQN", "is": "ICL", "de": "GSG", "it": "ISE", "sv": "SWL", "lt": "LLS", "en": "BFI"}
FAMILY = {"zh": "chinese", "uk": "french", "ru": "french", "bg": "french", "is": "french", "de": "german", "it": "french", "sv": "swedish", "lt": "french", "en": "banzsl"}
FAMILY_ZHANGHU = {"uk": "A", "ru": "A", "bg": "A", "is": "A", "it": "A", "lt": "A", "en": "B", "zh": "C", "de": "D", "sv": "E"}  # Zhang, Hu, Yu & Chen 2025 grouping
os.makedirs(OUT, exist_ok=True); os.makedirs(DATA, exist_ok=True)

def log(msg):
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"; print(line, flush=True)
    open(os.path.join(OUT, "run.log"), "a").write(line + "\n")

# ============================================================================ STAGE A: metadata and clips
def STAGE_A():
    """Fetch train/dev/test.json from the MLSLT/SP-10 GitHub repo (via the GitHub contents API) and download every clip
    from media.spreadthesign.com with 16 threads.  Writes OUT/sp10_index.json (one row per clip: split, id, lang, text, path)."""
    import base64, urllib.request
    from concurrent.futures import ThreadPoolExecutor
    meta = {}
    for part in ("train", "dev", "test"):
        p = os.path.join(DATA, f"{part}.json")
        if not os.path.exists(p):
            try:      # raw file first; the contents API (base64) as fallback
                data = urllib.request.urlopen(urllib.request.Request(f"https://raw.githubusercontent.com/MLSLT/SP-10/main/dataset/{part}.json", headers={"User-Agent": "sp10-grid"}), timeout=60).read()
                json.loads(data)
            except Exception:
                r = json.load(urllib.request.urlopen(urllib.request.Request(
                    f"https://api.github.com/repos/MLSLT/SP-10/contents/dataset/{part}.json", headers={"User-Agent": "sp10-grid"}), timeout=60))
                data = base64.b64decode(r["content"])
            open(p, "wb").write(data)
        meta[part] = json.load(open(p))
        if SMOKE:
            meta[part] = meta[part][:SMOKE]
    rows = []
    for part, items in meta.items():
        for obj in items:
            for s in obj["sign_list"]:
                if s.get("video_url"):
                    rows.append(dict(split=part, id=obj["id"], lang=s["lang"], text=s["text"], url=s["video_url"],
                                     path=os.path.join(DATA, "videos", part, str(obj["id"]), f"{s['lang']}.mp4")))
    log(f"index: {len(rows)} clips over {sum(len(v) for v in meta.values())} sentence ids; per language {dict(collections.Counter(r['lang'] for r in rows))}")
    def fetch(r):
        if os.path.exists(r["path"]) and os.path.getsize(r["path"]) > 0:
            return True
        os.makedirs(os.path.dirname(r["path"]), exist_ok=True)
        for attempt in range(3):
            try:
                req = urllib.request.Request(r["url"], headers={"User-Agent": "Mozilla/5.0"})
                data = urllib.request.urlopen(req, timeout=60).read()
                open(r["path"], "wb").write(data); return True
            except Exception as e:
                time.sleep(2 * (attempt + 1))
        return False
    with ThreadPoolExecutor(16) as ex:
        ok = list(ex.map(fetch, rows))
    for r, o in zip(rows, ok):
        r["downloaded"] = bool(o)
    log(f"downloaded {sum(ok)} of {len(rows)} clips")
    json.dump(rows, open(os.path.join(OUT, "sp10_index.json"), "w"), ensure_ascii=False)
    # English reference text per sentence id (the common target)
    ref = {f"{r['split']}/{r['id']}": r["text"] for r in rows if r["lang"] == "en"}
    json.dump(ref, open(os.path.join(OUT, "english_refs.json"), "w"), ensure_ascii=False, indent=0)
    log(f"english references for {len(ref)} sentence ids; stage A done")

# ============================================================================ STAGE B: poses
def _worker(args):
    import pilot_pipeline as pp
    items, mdir, out_path = args
    pose_lm, hand_lm = pp.make_landmarkers(mdir); res = {}
    for key, path in items:
        try:
            r = pp.extract_video(path, pose_lm, hand_lm, max_frames=200)
            if r is not None:
                seq, _ = pp.canonical_pose(*r); res[key] = seq
        except Exception as e:
            print("extract fail", key, e)
    np.savez_compressed(out_path, **res); return out_path

def STAGE_B(workers=4):
    """MediaPipe Tasks pose+hand landmarkers on every downloaded clip, canonical (T,55,3) form, one npz per language."""
    from multiprocessing import Pool
    mdir = os.path.join(DATA, "models"); os.makedirs(mdir, exist_ok=True)
    for u in ["https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task",
              "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"]:
        dst = os.path.join(mdir, u.rsplit("/", 1)[-1])
        if not os.path.exists(dst):
            subprocess.run(["curl", "-sL", "-o", dst, u], check=True)
    rows = [r for r in json.load(open(os.path.join(OUT, "sp10_index.json"))) if r["downloaded"]]
    for lang in LANGS:
        outp = os.path.join(OUT, f"poses_{lang}.npz")
        if os.path.exists(outp):
            continue
        items = [(f"{r['split']}/{r['id']}", r["path"]) for r in rows if r["lang"] == lang]
        chunks = [items[i::workers] for i in range(workers)]
        parts = [(c, mdir, outp.replace(".npz", f".part{i}.npz")) for i, c in enumerate(chunks) if c]
        t = time.time()
        with Pool(len(parts)) as pool:
            done = pool.map(_worker, parts)
        res = {}
        for p in done:
            z = np.load(p); res.update({k: z[k] for k in z.files}); os.remove(p)
        np.savez_compressed(outp, **res)
        log(f"{lang} ({SIGN[lang]}): poses for {len(res)} of {len(items)} clips in {round(time.time()-t)} s")
    audit = {}
    for lang in LANGS:
        z = np.load(os.path.join(OUT, f"poses_{lang}.npz")); n = fr = hp = 0
        for k in z.files:
            s = z[k]; n += 1; fr += len(s); hp += np.isfinite(s[:, 13:, 0]).any(1).sum()
        audit[lang] = {"clips": n, "mean_frames": round(fr / max(n, 1), 1), "frames_with_a_hand": round(hp / max(fr, 1), 3)}
    json.dump(audit, open(os.path.join(OUT, "pose_audit.json"), "w"), indent=1); log(f"audit {audit}"); log("stage B done")

# ============================================================================ STAGE C: similarity before training
def STAGE_C():
    """S1 kinematic DTW over the shared TRAIN sentences (the paper's S1), S1b handshape MMD, S1c location MMD, S2 family
    labels (two classifications).  45 unordered pairs.  Written to OUT/similarity.json before any grid run exists."""
    import pilot_pipeline as pp
    poses = {l: np.load(os.path.join(OUT, f"poses_{l}.npz")) for l in LANGS}
    train_keys = {l: {k for k in poses[l].files if k.startswith("train/")} for l in LANGS}
    hand, loc, seqs = {}, {}, {}
    for l in LANGS:
        H, Lc, S = [], [], {}
        for k in sorted(train_keys[l]):
            s = poses[l][k]; H.append(pp.hand_shape_vectors(s)); Lc.append(pp.location_vectors(s))
            s48, _ = pp.resample(s, np.ones((len(s), 3), bool)); S[k] = s48      # NaNs kept; DTW masks missing joints (audit 2026-09-22)
        hand[l] = np.concatenate(H); loc[l] = np.concatenate(Lc); loc[l] = loc[l][np.isfinite(loc[l]).all(1)]; seqs[l] = S
        log(f"{l}: {len(S)} train clips, {len(hand[l])} hand frames")
    sim = {}
    for i, a in enumerate(LANGS):
        for b in LANGS[i + 1:]:
            shared = sorted(set(seqs[a]) & set(seqs[b]))
            d = [pp.dtw_distance(seqs[a][k], seqs[b][k], np.isfinite(seqs[a][k][..., 0]), np.isfinite(seqs[b][k][..., 0])) for k in shared]
            sim[f"{a}|{b}"] = {"S1_dtw": float(np.mean(d)), "S1_n_sentences": len(d), "S1b_hand_mmd": float(pp.mmd_rbf(hand[a], hand[b])),
                               "S1c_loc_mmd": float(pp.mmd_rbf(loc[a], loc[b])), "S2_same_family_hist": int(FAMILY[a] == FAMILY[b]),
                               "S2_same_family_zhanghu": int(FAMILY_ZHANGHU[a] == FAMILY_ZHANGHU[b])}
            log(f"{SIGN[a]}-{SIGN[b]}: {sim[f'{a}|{b}']}")
    json.dump(sim, open(os.path.join(OUT, "similarity.json"), "w"), indent=1)
    log("stage C done — similarity fixed before any training")

# ============================================================================ STAGE D: the grid (one seed per run)
def _load_all():
    import pilot_pipeline as pp
    src = IN_POSES or OUT
    poses = {l: np.load(os.path.join(src, f"poses_{l}.npz")) for l in LANGS}
    refs = json.load(open(os.path.join(src, "english_refs.json")))
    data = {}
    for l in LANGS:
        for split in ("train", "dev", "test"):
            X, Y, K = [], [], []
            for k in sorted(poses[l].files):
                if not k.startswith(split + "/") or k not in refs:
                    continue
                s48, _ = pp.resample(poses[l][k], np.ones((len(poses[l][k]), 3), bool))
                X.append(pp.fill_nan_zero(s48)); Y.append(refs[k]); K.append(k)
            data[(l, split)] = (np.stack(X).astype(np.float32), Y, K)
    return data

def build_slt_model(d=256):
    """Pose encoder (4-layer transformer over per-frame joint vectors) feeding a T5-small decoder via its encoder-hidden
    interface; the decoder is text-initialised (the paper's design), the encoder is trained from scratch per cell."""
    import torch, torch.nn as nn
    from transformers import T5ForConditionalGeneration, T5TokenizerFast
    tok = T5TokenizerFast.from_pretrained("t5-small")
    t5 = T5ForConditionalGeneration.from_pretrained("t5-small")
    dm = t5.config.d_model
    class PoseT5(nn.Module):
        def __init__(self):
            super().__init__()
            self.inp = nn.Linear(55 * 4, dm); self.pos = nn.Parameter(torch.zeros(1, 48, dm))
            enc = nn.TransformerEncoderLayer(dm, 8, 4 * dm, 0.2, batch_first=True); self.enc = nn.TransformerEncoder(enc, 4)
            self.norm = nn.LayerNorm(dm); self.t5 = t5
        def encode(self, x):
            return self.norm(self.enc(self.inp(x.flatten(2)) + self.pos))
        def forward(self, x, labels):
            from transformers.modeling_outputs import BaseModelOutput
            return self.t5(encoder_outputs=BaseModelOutput(last_hidden_state=self.encode(x)), labels=labels).loss
        def generate(self, x, **kw):
            from transformers.modeling_outputs import BaseModelOutput
            return self.t5.generate(encoder_outputs=BaseModelOutput(last_hidden_state=self.encode(x)), max_new_tokens=32, num_beams=4, **kw)
    return PoseT5(), tok

def train_slt(model, tok, Xtr, Ytr, Xdv, Ydv, epochs, seed, device, lr=3e-4, batch=16):
    import torch, pilot_pipeline as pp
    torch.manual_seed(seed); rng = random.Random(seed); model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * math.ceil(len(Xtr) / batch); sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps)
    Xt = torch.tensor(Xtr); lab = tok(Ytr, padding=True, truncation=True, max_length=32, return_tensors="pt").input_ids
    lab[lab == tok.pad_token_id] = -100
    best, best_state = -1.0, None
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xt))
        for i in range(0, len(perm), batch):
            idx = perm[i:i + batch]; xb = pp.augment(Xt[idx], rng).to(device)
            loss = model(xb, lab[idx].to(device)); opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
        if (ep + 1) % 5 == 0 or ep == epochs - 1:
            score = chrf_score(decode(model, tok, Xdv, device), Ydv)
            if score > best:
                best, best_state = score, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state); return best

def decode(model, tok, X, device, batch=32):
    import torch
    model.eval(); outs = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            ids = model.generate(torch.tensor(X[i:i + batch]).to(device)); outs += tok.batch_decode(ids, skip_special_tokens=True)
    return outs

def chrf_score(hyps, refs):
    from sacrebleu.metrics import CHRF
    return CHRF().corpus_score(hyps, [refs]).score

def score_all(hyps, refs):
    """BLEU and chrF in-run (sacreBLEU).  BLEURT is scored afterwards from the saved hypotheses (STAGE_F), because the
    BLEURT package pins an older transformers than the training stage uses."""
    from sacrebleu.metrics import BLEU, CHRF
    return {"bleu": BLEU().corpus_score(hyps, [refs]).score, "chrf": CHRF().corpus_score(hyps, [refs]).score, "bleurt": float("nan")}

def save_hyps(cond, s, t, hyps, keys):
    hd = os.path.join(OUT, f"hyps_seed{SEED}"); os.makedirs(hd, exist_ok=True)
    json.dump({"condition": cond, "source": s, "target": t, "seed": SEED, "keys": keys, "hyps": hyps},
              open(os.path.join(hd, f"{cond}_{s}_{t}.json"), "w"), ensure_ascii=False)

def STAGE_D():
    """One seed: 10 from-scratch models (also the pretrained sources), then 90 fine-tunes.  Every cell appends to
    OUT/grid_seed{SEED}.csv with BLEU, chrF, BLEURT on the target's test split; resumable."""
    import torch, csv
    device = "cuda" if torch.cuda.is_available() else "cpu"
    data = _load_all()
    gpath = os.path.join(OUT, f"grid_seed{SEED}.csv")
    done = set()
    if os.path.exists(gpath):
        done = {(r["condition"], r["source"], r["target"]) for r in csv.DictReader(open(gpath))}
    else:
        open(gpath, "w").write("condition,source,target,seed,dev_chrf,test_bleu,test_chrf,test_bleurt,epochs\n")
    def rec(cond, s, t, dev, sc):
        open(gpath, "a").write(f"{cond},{s},{t},{SEED},{dev:.3f},{sc['bleu']:.3f},{sc['chrf']:.3f},{sc['bleurt']:.3f},{EPOCHS}\n")
        log(f"{cond} {SIGN[s]}->{SIGN[t]}: dev chrF {dev:.2f} test BLEU {sc['bleu']:.2f} chrF {sc['chrf']:.2f} BLEURT {sc['bleurt']:.2f}")
    for l in LANGS:                                            # scratch = source model for l
        ck = os.path.join(OUT, f"src_{l}_seed{SEED}.pt")
        if ("scratch", l, l) in done and os.path.exists(ck):
            continue
        # Audit 2026-10-03: seed MUST be set before model construction, not after -- otherwise the
        # first language's init (zh, per LANGS order) draws from whatever ambient/leftover RNG state
        # preceded this stage, which is not a reproducible function of SEED. Reseeding fresh immediately
        # before every build_slt_model() call (not just the first) also removes the order-dependence
        # that a bare reorder would leave: each language's scratch model now starts from the identical,
        # SEED-determined initial weights, so only the training data differs between cells -- the same
        # fix already applied to transfer_head() in pilot_pipeline.py, backported here.
        torch.manual_seed(SEED)
        model, tok = build_slt_model()
        dev = train_slt(model, tok, *data[(l, "train")][:2], *data[(l, "dev")][:2], EPOCHS, SEED, device)
        torch.save(model.state_dict(), ck)
        hyps = decode(model, tok, data[(l, "test")][0], device); save_hyps("scratch", l, l, hyps, data[(l, "test")][2])
        rec("scratch", l, l, dev, score_all(hyps, data[(l, "test")][1]))
    for s in LANGS:
        for t in LANGS:
            if s == t or ("transfer", s, t) in done:
                continue
            model, tok = build_slt_model(); model.load_state_dict(torch.load(os.path.join(OUT, f"src_{s}_seed{SEED}.pt")))
            dev = train_slt(model, tok, *data[(t, "train")][:2], *data[(t, "dev")][:2], EPOCHS, SEED, device)
            hyps = decode(model, tok, data[(t, "test")][0], device); save_hyps("transfer", s, t, hyps, data[(t, "test")][2])
            rec("transfer", s, t, dev, score_all(hyps, data[(t, "test")][1]))
    log(f"stage D seed {SEED} done")

# ============================================================================ STAGE E: analysis (after all seeds)
def STAGE_E(grid_files, similarity_path):
    """Join gains (transfer minus scratch, same target and seed) with similarities; mixed model per metric."""
    import pandas as pd, statsmodels.formula.api as smf, warnings; warnings.filterwarnings("ignore")
    from scipy import stats
    g = pd.concat([pd.read_csv(f) for f in grid_files]); sim = json.load(open(similarity_path))
    bl = os.path.join(OUT, "bleurt.csv")
    if os.path.exists(bl):
        b = pd.read_csv(bl); g = g.drop(columns=["test_bleurt"]).merge(b, on=["condition", "source", "target", "seed"], how="left")
    rows = []
    for _, r in g[g.condition == "transfer"].iterrows():
        sc = g[(g.condition == "scratch") & (g.target == r.target) & (g.seed == r.seed)].iloc[0]
        key = f"{r.source}|{r.target}" if f"{r.source}|{r.target}" in sim else f"{r.target}|{r.source}"; s = sim[key]
        rows.append(dict(source=r.source, target=r.target, seed=int(r.seed), gain_bleurt=r.test_bleurt - sc.test_bleurt, gain_chrf=r.test_chrf - sc.test_chrf,
                         gain_bleu=r.test_bleu - sc.test_bleu, S1=s["S1_dtw"], S1b=s["S1b_hand_mmd"], S1c=s["S1c_loc_mmd"],
                         fam_hist=s["S2_same_family_hist"], fam_zh=s["S2_same_family_zhanghu"]))
    df = pd.DataFrame(rows); df.to_csv(os.path.join(OUT, "pairs.csv"), index=False)
    res = {}
    for metric in ("gain_bleurt", "gain_chrf", "gain_bleu"):
        if df[metric].isna().all():
            continue
        d = df.dropna(subset=[metric]).copy()
        for col in ("S1", "S1b", "S1c"):
            if not (d[col].std() > 0):
                continue
            d[col + "_z"] = (d[col] - d[col].mean()) / d[col].std()
            m = smf.mixedlm(f"{metric} ~ {col}_z", d, groups=d["target"], re_formula="1").fit(reml=False)
            res[f"{metric}~{col}"] = {"coef": float(m.params[col + "_z"]), "ci": [float(x) for x in m.conf_int().loc[col + "_z"]], "p": float(m.pvalues[col + "_z"])}
        for col in ("fam_hist", "fam_zh"):
            if d[col].nunique() < 2:
                continue
            m = smf.mixedlm(f"{metric} ~ {col}", d, groups=d["target"], re_formula="1").fit(reml=False)
            res[f"{metric}~{col}"] = {"coef": float(m.params[col]), "ci": [float(x) for x in m.conf_int().loc[col]], "p": float(m.pvalues[col])}
        wt = {}
        for t, sub in d.groupby(["source", "target"]).mean(numeric_only=True).reset_index().groupby("target"):
            wt[t] = float(stats.spearmanr(sub["S1"], sub[metric]).statistic)
        res[f"{metric}~S1_within_target_spearman"] = wt
    json.dump(res, open(os.path.join(OUT, "readings.json"), "w"), indent=1); log(json.dumps(res, indent=1)); log("stage E done")

# ============================================================================ STAGE F: BLEURT from saved hypotheses
def STAGE_F(hyps_dirs, refs_path, out_csv):
    """Run in an environment with transformers<5 and bleurt-pytorch (pip install transformers==4.44.2
    git+https://github.com/lucadiliello/bleurt-pytorch.git).  Scores every saved hypothesis file with BLEURT-20-D12."""
    import torch, glob, csv
    from bleurt_pytorch import BleurtForSequenceClassification, BleurtTokenizer
    refs = json.load(open(refs_path))
    m = BleurtForSequenceClassification.from_pretrained("lucadiliello/BLEURT-20-D12").eval(); t = BleurtTokenizer.from_pretrained("lucadiliello/BLEURT-20-D12")
    w = csv.writer(open(out_csv, "w")); w.writerow(["condition", "source", "target", "seed", "test_bleurt"])
    for hd in hyps_dirs:
        for f in sorted(glob.glob(os.path.join(hd, "*.json"))):
            h = json.load(open(f)); R = [refs[k] for k in h["keys"]]; sc = []
            with torch.no_grad():
                for i in range(0, len(R), 32):
                    inp = t(R[i:i + 32], h["hyps"][i:i + 32], padding="longest", truncation=True, max_length=128, return_tensors="pt")
                    sc += m(**inp).logits.flatten().tolist()
            w.writerow([h["condition"], h["source"], h["target"], h["seed"], round(100 * float(np.mean(sc)), 3)])
    log(f"stage F done -> {out_csv}")

if __name__ == "__main__":
    for s in sys.argv[1:]:
        globals()[f"STAGE_{s}"]()
''')
h = hashlib.sha256(open('sp10_driver.py','rb').read()).hexdigest(); print('sp10_driver.py', h); assert h == '234c59835f23db6686375c3fe6a735c807f107bc8efaf6059a0a5fc32c1ecfe7', 'file differs from the reviewed copy'
open('sp10_fetch_rest.py','w').write(r'''"""
sp10_fetch_rest.py -- recover the SP-10 clips that Run 2 never downloaded, extract their poses, merge them with the
poses Run 2 already has, and recompute the similarity matrix on the complete data.

Why this exists.  The Run 2 log (pilot/sp10_out/run.log) says 'downloaded 6858 of 11741 clips'.  4,883 clips (42%) were
never fetched: 478 of 1,186 sentence ids have no clip in any language.  Every later stage (similarity, the 90-pair grid,
the paper's numbers) used only the clips that arrived.  A HEAD check on 12 of the missing URLs returned HTTP 200, so the
failure looks transient.  This script re-fetches only the missing clips.

Three stages, run as two Kaggle notebooks so that each ends with a saved, checkable output:
  D1_download   (notebook 1, CPU, Internet ON)   re-fetch the missing clips, with verification, politeness and a report.
                Writes clips/, sp10_index_v2.json, fetch_report.json.
  D2_extract    (notebook 2, CPU, Internet ON)   MediaPipe poses for the NEW clips only, merged with Run 2's poses.
  D3_finish     (notebook 2, CPU)                pose audit, similarity on all train clips, counts table, sha256 manifest.

Nothing from Run 2 is modified: its dataset is read-only input and every file written here goes to a new output folder.
Everything resumes: re-running a stage skips clips and languages that already exist.

Inputs it looks for under /kaggle/input (override the root with SP10_INPUT for a test):
  notebook 1: Run 2's output   (found by its sp10_index.json)
  notebook 2: Run 2's output   (found by poses_zh.npz)  AND  notebook 1's output (found by sp10_index_v2.json)
"""
import os, sys, json, glob, time, hashlib, collections, subprocess, urllib.request
from concurrent.futures import ThreadPoolExecutor
import numpy as np

sys.path.insert(0, ".")
# D1 needs nothing from the driver, so notebook 1 ships this file alone.  D2 and D3 import sp10_driver themselves
# (LANGS, SIGN, _worker, STAGE_C), and both read the same SP10_OUT / SP10_DATA environment variables.
OUT = os.environ.get("SP10_OUT", "/kaggle/working/sp10_out")       # all new files go here (the saved Kaggle output)
DATA = os.environ.get("SP10_DATA", "/tmp/sp10_data")
os.makedirs(OUT, exist_ok=True); os.makedirs(DATA, exist_ok=True)
CLIPS = os.path.join(OUT, "clips")                # downloaded videos: clips/<split>/<id>/<lang>.mp4
INPUT = os.environ.get("SP10_INPUT", "/kaggle/input")
SMOKE = int(os.environ.get("SP10_FETCH_SMOKE", "0"))   # >0: only this many missing clips (pipeline test)


def log(msg):
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"; print(line, flush=True)
    open(os.path.join(OUT, "run.log"), "a").write(line + "\n")


def find_one(name):
    """The single file called name under the input root; refuses to guess if there are zero or several."""
    hits = sorted(glob.glob(os.path.join(INPUT, "**", name), recursive=True))
    if len(hits) != 1:
        raise SystemExit(f"expected exactly one {name} under {INPUT}, found {len(hits)}: {hits[:5]} -- attach the right notebook outputs")
    return hits[0]


# ============================================================================ D1: download the missing clips
def _fetch_one(row, tries=6):
    """One clip.  A clip counts as fetched only if the HTTP read is complete (matches Content-Length when the server
    sends it), is large enough, and looks like an MP4 ('ftyp' or 'moov' box in the first 4 KB).  Written to .part then renamed, so a
    killed run never leaves a half file that looks finished.  Backoff doubles per try (3, 6, 12, ... capped at 60 s).
    Returns (ok, error text).  Run 2 swallowed every exception; this keeps the reason."""
    dst = os.path.join(CLIPS, row["split"], str(row["id"]), f"{row['lang']}.mp4")
    if os.path.exists(dst) and os.path.getsize(dst) > 1000:
        return True, None
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    err = None
    for a in range(tries):
        try:
            req = urllib.request.Request(row["url"], headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=60) as r:
                data = r.read()
                want = r.headers.get("Content-Length")
            if want and len(data) != int(want):
                raise IOError(f"short read {len(data)} of {want}")
            if len(data) < 1000 or (b"ftyp" not in data[:4096] and b"moov" not in data[:4096]):
                raise IOError(f"not an mp4 ({len(data)} bytes)")
            tmp = dst + ".part"
            with open(tmp, "wb") as f:
                f.write(data)
            os.replace(tmp, dst)
            return True, None
        except Exception as e:
            err = f"{type(e).__name__}: {str(e)[:120]}"
            time.sleep(min(60, 3 * 2 ** a))
    return False, err


def _run_pass(items, workers, tag, batch=300):
    """Fetch the items in batches.  After each batch the failure rate is logged; if more than half of a batch failed the
    server is probably throttling us, so pause 3 minutes and halve the workers (min 1) instead of hammering it."""
    errs, ok_n = {}, 0
    for i in range(0, len(items), batch):
        chunk = items[i:i + batch]
        with ThreadPoolExecutor(workers) as ex:
            res = list(ex.map(_fetch_one, chunk))
        bad = [(r, e) for r, (o, e) in zip(chunk, res) if not o]
        ok_n += len(chunk) - len(bad)
        for r, e in bad:
            errs[r["url"]] = e
        log(f"[{tag}] {i + len(chunk)}/{len(items)} done; batch failures {len(bad)}/{len(chunk)}; workers {workers}")
        if len(chunk) >= 20 and len(bad) > len(chunk) / 2 and i + batch < len(items):
            log(f"[{tag}] more than half the batch failed -> pause 180 s, workers {workers} -> {max(1, workers // 2)}")
            time.sleep(180); workers = max(1, workers // 2)
    return ok_n, errs


def D1_download():
    old_idx = find_one("sp10_index.json")
    rows = json.load(open(old_idx))
    for r in rows:
        r["source"] = "run2" if r["downloaded"] else "pending"
    todo = [r for r in rows if not r["downloaded"]]
    if SMOKE:
        todo = todo[:SMOKE]
    log(f"index {len(rows)} clips; Run 2 had {sum(r['downloaded'] for r in rows)}; to fetch {len(todo)}")

    # three passes: 6 workers, then the failures again after 2 minutes with 2 workers, then after 5 minutes with 1
    report = {"to_fetch": len(todo), "passes": []}
    pending, final_errs = todo, {}
    for tag, workers, wait in (("pass1", 6, 0), ("pass2", 2, 120), ("pass3", 1, 300)):
        if not pending:
            break
        if wait:
            log(f"waiting {wait} s before {tag} on {len(pending)} failures"); time.sleep(wait)
        ok_n, errs = _run_pass(pending, workers, tag)
        report["passes"].append({"pass": tag, "attempted": len(pending), "fetched": ok_n, "failed": len(errs)})
        pending = [r for r in pending if r["url"] in errs]
        final_errs = errs

    # write index v2: paths are relative to this output folder so notebook 2 can find the clips wherever it is mounted
    for r in todo:
        p = os.path.join("clips", r["split"], str(r["id"]), f"{r['lang']}.mp4")
        full = os.path.join(OUT, p)
        if os.path.exists(full) and os.path.getsize(full) > 1000:
            r["downloaded"], r["source"], r["path"] = True, "new", p
        else:
            r["source"] = "missing"
    json.dump(rows, open(os.path.join(OUT, "sp10_index_v2.json"), "w"), ensure_ascii=False)

    n_new = sum(r["source"] == "new" for r in rows)
    report["fetched_new"] = n_new
    report["still_missing"] = sum(r["source"] == "missing" for r in rows)
    report["total_with_clip"] = sum(r["downloaded"] for r in rows)
    report["failure_reasons"] = dict(collections.Counter(str(e)[:60] for e in final_errs.values()))
    report["failed_urls"] = [r["url"] for r in pending]
    by = collections.Counter((r["split"], r["downloaded"]) for r in rows)
    report["by_split"] = {s: {"have": by[(s, True)], "lack": by[(s, False)]} for s in ("train", "dev", "test")}
    json.dump(report, open(os.path.join(OUT, "fetch_report.json"), "w"), indent=1)
    log(json.dumps({k: v for k, v in report.items() if k != "failed_urls"}))
    log("D1 done")


# ============================================================================ D2: poses for the new clips, merged
def D2_extract(workers=4):
    from multiprocessing import Pool
    import sp10_driver as d
    idx_path = find_one("sp10_index_v2.json")
    base = os.path.dirname(idx_path)
    old_dir = os.path.dirname(find_one("poses_zh.npz"))
    rows = json.load(open(idx_path))
    mdir = os.path.join(DATA, "models"); os.makedirs(mdir, exist_ok=True)
    for u in ["https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task",
              "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"]:
        dst = os.path.join(mdir, u.rsplit("/", 1)[-1])
        if not os.path.exists(dst):
            subprocess.run(["curl", "-sL", "-o", dst, u], check=True)
    failures = {}
    for lang in d.LANGS:
        merged_p = os.path.join(OUT, f"poses_{lang}.npz")
        new_p = os.path.join(OUT, f"poses_new_{lang}.npz")
        if os.path.exists(merged_p):
            continue
        items = [(f"{r['split']}/{r['id']}", os.path.join(base, r["path"])) for r in rows if r["lang"] == lang and r["source"] == "new"]
        t = time.time()
        if os.path.exists(new_p):                        # extraction finished earlier; only the merge was lost
            new = dict(np.load(new_p))
        else:
            chunks = [items[i::workers] for i in range(workers)]
            parts = [(c, mdir, new_p.replace(".npz", f".part{i}.npz")) for i, c in enumerate(chunks) if c]
            new = {}
            if parts:
                with Pool(len(parts)) as pool:
                    done = pool.map(d._worker, parts)
                for p in done:
                    z = np.load(p); new.update({k: z[k] for k in z.files}); os.remove(p)
            np.savez_compressed(new_p, **new)            # new-only copy is kept as its own output file
        old = np.load(os.path.join(old_dir, f"poses_{lang}.npz"))
        overlap = set(new) & set(old.files)
        assert not overlap, f"{lang}: {len(overlap)} keys are in both Run 2 poses and the new poses"
        res = {k: old[k] for k in old.files}; res.update(new)
        tmp = merged_p + ".tmp.npz"
        np.savez_compressed(tmp, **res); os.replace(tmp, merged_p)
        expect = sum(1 for r in rows if r["lang"] == lang and r["downloaded"])
        lost = [k for k, _ in items if k not in new]
        failures[lang] = lost
        log(f"{lang} ({d.SIGN[lang]}): old {len(old.files)} + new {len(new)} = {len(res)} poses; clips in index {expect}; "
            f"new clips without a pose {len(lost)}; {round(time.time() - t)} s")
    if failures:
        json.dump(failures, open(os.path.join(OUT, "extract_failures.json"), "w"), indent=1)
    log("D2 done")


# ============================================================================ D3: audit, similarity, counts, manifest
def D3_finish():
    import sp10_driver as d
    idx_path = find_one("sp10_index_v2.json")
    old_dir = os.path.dirname(find_one("poses_zh.npz"))
    rows = json.load(open(idx_path))
    # English references: Run 2's file, checked against the index so a mismatch cannot pass silently
    refs = json.load(open(os.path.join(old_dir, "english_refs.json")))
    from_idx = {f"{r['split']}/{r['id']}": r["text"] for r in rows if r["lang"] == "en"}
    assert refs == from_idx, "english_refs.json from Run 2 does not match the index"
    json.dump(refs, open(os.path.join(OUT, "english_refs.json"), "w"), ensure_ascii=False, indent=0)

    audit, counts = {}, {}
    for lang in d.LANGS:
        z = np.load(os.path.join(OUT, f"poses_{lang}.npz")); n = fr = hp = 0
        cs = collections.Counter(k.split("/")[0] for k in z.files)
        for k in z.files:
            s = z[k]; n += 1; fr += len(s); hp += np.isfinite(s[:, 13:, 0]).any(1).sum()
        audit[lang] = {"clips": n, "mean_frames": round(fr / max(n, 1), 1), "frames_with_a_hand": round(hp / max(fr, 1), 3)}
        counts[lang] = dict(cs)
    json.dump(audit, open(os.path.join(OUT, "pose_audit.json"), "w"), indent=1)
    json.dump(counts, open(os.path.join(OUT, "clip_counts.json"), "w"), indent=1)
    log(f"clips per language and split: {counts}")
    log(f"audit {audit}")

    d.STAGE_C()                                          # S1 DTW etc. on ALL train clips now present, before any training

    man = {}                                             # sha256 of every file we wrote (clips/ excluded; notebook 1 holds them)
    for p in sorted(glob.glob(os.path.join(OUT, "*"))):
        if os.path.isfile(p):
            h = hashlib.sha256(open(p, "rb").read()).hexdigest()
            man[os.path.basename(p)] = {"bytes": os.path.getsize(p), "sha256": h}
    json.dump(man, open(os.path.join(OUT, "manifest.json"), "w"), indent=1)
    log("D3 done; manifest has " + str(len(man)) + " files")
''')
h = hashlib.sha256(open('sp10_fetch_rest.py','rb').read()).hexdigest(); print('sp10_fetch_rest.py', h); assert h == '8769c19b0521604eab4a0e8431904981da4866a30dede738e56e24b915800b12', 'file differs from the reviewed copy'


In [ ]:
import subprocess, sys; subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mediapipe==0.10.32'], check=False)
sys.path.insert(0, '.'); import sp10_fetch_rest as f
f.D2_extract(workers=4)

In [ ]:
f.D3_finish()

In [ ]:
import json; print(open(f.OUT + '/clip_counts.json').read()); print(open(f.OUT + '/pose_audit.json').read())